### 엑셀에서 데이터 다루기

In [6]:
import win32com.client

- 엑셀 프로그램 실행

In [7]:
# 몇 월인지 확인
mon = input('몇월입니까?')

In [8]:
excel = win32com.client.Dispatch('Excel.Application')
excel.Visible = True

- 워크북/시트 불러오기

In [9]:
file_name = 'D:/LAB/RPA/data.xlsx'
wb = excel.Workbooks.Open(file_name)
ws = wb.Worksheets(mon + '월 판매(계획비)')

- 데이터 추출

In [10]:
k6 = ws.Cells(13, 11).Value
print(k6)

2900.0


In [11]:
s6 = ws.Range("k14").Value
print(s6)

20950.0


### P-GPT API Key를 활용한 분석 텍스트 생성

In [23]:
import os
from pathlib import Path

from openai import OpenAI


def load_env(path=".env"):
    env_path = Path(path)
    if not env_path.exists():
        return

    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env()

api_key = os.getenv("AIGPT_API_KEY")
if not api_key:
    raise RuntimeError("AIGPT_API_KEY is not set in .env")

client = OpenAI(
    api_key=api_key,
    base_url="http://aigpt.posco.net/gpgpta01-gpt/v1"
)

analysis_data = """
- 생산량: 전월 대비 3.2% 증가
- 판매량: 전월 대비 1.8% 감소
- 매출액: 계획 대비 95% 달성
- 주요 이슈: 원재료 단가 상승, 일부 제품 출하 지연
""".strip()

prompt = f"""
요구사항:
- 핵심 원인과 영향을 구분해서 분석
- 숫자 변화가 의미하는 경영적 시사점 포함
- 3~5문장으로 간결하게 작성
- 보고서 문체로 작성

분석 데이터:
{analysis_data}
""".strip()

response = client.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {
            "role": "system",
            "content": "당신은 경영실적 보고서용 시사점을 작성하는 데이터 분석 전문가입니다."
        },
        {"role": "user", "content": prompt}
    ]
)

insight_text = response.choices[0].message.content.strip()
print(insight_text)


생산량은 전월 대비 3.2% 증가했으나, 일부 제품의 출하 지연과 판매량 1.8% 감소가 맞물리며 매출 실현으로의 전환이 원활하지 못한 것으로 분석된다. 특히 매출액이 계획 대비 95%에 그친 것은 생산 확대에도 불구하고 판매·출하 운영의 비효율이 실적 저하로 이어졌음을 시사한다. 또한 원재료 단가 상승은 수익성 부담을 가중시키는 핵심 원인으로, 향후 동일 매출 수준에서도 채산성 저하 가능성이 확대될 우려가 있다. 이에 따라 출하 안정화 및 판매 회복 대책과 함께 원가 상승분에 대한 선제적 대응이 필요하다.


### 파워포인트에 데이터 입력

In [ ]:
from pptx import Presentation
from pptx.enum.text import PP_ALIGN
from pptx.util import Pt

pptx_path = r"temp.pptx"
prs = Presentation(pptx_path)

slide = prs.slides[0]

def set_table_cell_font(cell, font_name='맑은 고딕', font_size=11):
    for paragraph in cell.text_frame.paragraphs:
        for run in paragraph.runs:
            run.font.name = font_name
            run.font.size = Pt(font_size)



# 제목 입력
for shape in slide.shapes:
    if shape.name == "TextBox 18":
        
        # 기존 텍스트 변경
        shape.text = "여기는 제목 입력입니다."

        # Center align and set font size
        for paragraph in shape.text_frame.paragraphs:
            paragraph.alignment = PP_ALIGN.CENTER
            for run in paragraph.runs:
                run.font.name = "HY\uacac\uace0\ub515"
                run.font.size = Pt(24)

        # 또는
        # shape.text_frame.text = "새로운 텍스트 입력"

        print("제목 입력 완료")



# 생산.판매 실적 분석내용 입력
for shape in slide.shapes:
    if shape.name == '직사각형 31':
        shape.text = '여기는 생산.판매 실적 분석내용을 입력하는 곳입니다.'
        for paragraph in shape.text_frame.paragraphs:
            for run in paragraph.runs:
                run.font.size = Pt(11)
        print("Insight text input complete")
        break

# 경영 실적 및 주요 관리지표 분석내용 입력
for shape in slide.shapes:
    if shape.name == '직사각형 32':
        shape.text = '여기는 경영 실적 및 주요 관리지표 분석내용을 입력하는 곳입니다.'
        for paragraph in shape.text_frame.paragraphs:
            for run in paragraph.runs:
                run.font.size = Pt(11)
        print("Insight text input complete")
        break


# 엑셀에서 추출한 데이터를 표에 입력 
table_38_count = 0

for shape in slide.shapes:

    if shape.name == "표 38":

        if shape.has_table:
            table_38_count += 1

            table = shape.table

            # python-pptx table indexes start at 0
            if table_38_count == 1: #첫번째 표 순서
                for row_idx in range(3, 6): #행(3행부터 시작해야 맞음)
                    for col_idx in range(2, 8): #열
                        cell = table.cell(row_idx, col_idx)
                        cell.text = str(k6)
                        set_table_cell_font(cell)
                print("첫 번째 표 입력 완료")
            elif table_38_count == 2: #2번째 표
                for row_idx in range(2, 8):#행
                    for col_idx in range(2, 8):#열
                        cell = table.cell(row_idx, col_idx)
                        cell.text = str(s6)
                        set_table_cell_font(cell)
                print("두 번째 표 입력 완료")
                break


# 저장
#prs.save(pptx_path)
prs.save(r"result.pptx")

제목 입력 완료
Insight text input complete
Insight text input complete
첫 번째 표 입력 완료
두 번째 표 입력 완료
